# Tool Use と RAG

RAG は外部文書を検索して根拠を回答へ接続する設計です。Tool Use は計算、検索、Web 操作、DB 問い合わせのような外部機能を明示的に呼び出す設計です。どちらも、モデル単体で推測し続けるのではなく、必要な情報や能力を外へ取りに行くために使います。

この教材では、まず「文書を読ませる問題」と「外部手続きを実行させる問題」を分けます。そのうえで、検索単位、再ランキング、根拠の対応、ツール選択の失敗を別々に観察し、どこを直せば最終回答が良くなるのかを追います。

## RAG と Tool Use を分けて読む

RAG は「読む」接続です。質問に合う文書片を取得し、回答と根拠の対応を残します。Tool Use は「動かす」接続です。計算機、HTML 解析、API 呼び出しのような手続きを外部へ委譲します。実務では、検索の失敗、引用のずれ、ツール選択の誤りを別々に測る必要があります。

境界を決める基準は、回答に必要なのが文書中の根拠なのか、外部手続きの実行結果なのかです。文書根拠なら RAG、計算や操作なら Tool Use に寄せます。この分離により、検索品質、ツール成功率、最終回答品質を別々に改善できます。

In [ ]:
import ast
import math
import operator as op
import re
from collections import Counter, defaultdict
from html.parser import HTMLParser


def tokenize(text):
    text = re.sub(r'\s+', '', text.lower())
    if not text:
        return []
    if len(text) == 1:
        return [text]
    return [text[i:i+2] for i in range(len(text)-1)]


def cosine(a, b):
    keys = set(a) | set(b)
    dot = sum(a.get(k, 0.0) * b.get(k, 0.0) for k in keys)
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return 0.0 if na == 0 or nb == 0 else dot / (na * nb)

## 文書を検索単位に分ける

検索単位が大きすぎると、関係の薄い文まで文脈に混ざります。小さすぎると、答えに必要な情報が分断されます。文単位と固定長を混ぜ、後で重複除去と再ランキングを入れると、検索の粗さを観察しやすくなります。

チャンク設計は検索性能と回答の根拠対応を同時に左右します。短いチャンクは狙った語を拾いやすい一方で、文脈が欠けやすくなります。長いチャンクは文脈を保ちますが、関係の薄い語も混ざります。

In [ ]:
documents = [
    {
        'id': 'rl',
        'title': '強化学習',
        'text': 'ベルマン最適方程式は最適価値関数を再帰的に定義する。価値反復法はこの更新を繰り返し、最適方策を求める。',
    },
    {
        'id': 'finetune',
        'title': 'ファインチューニング',
        'text': 'SFTは指示と回答ペアで応答スタイルを調整する。LoRAは低ランク行列だけを更新し、学習コストを下げる。',
    },
    {
        'id': 'rag',
        'title': 'RAG 実装',
        'text': 'RAGではチャンク化、検索、再ランキング、引用付き生成を分けて評価する。検索精度が低いと回答品質も下がる。',
    },
    {
        'id': 'guardrails',
        'title': 'ガードレール',
        'text': 'Input Railsは危険入力を検知して遮断する。Output Railsは生成結果を検査し、ポリシー違反を抑える。',
    },
]


def sentence_chunks(text):
    return [p for p in re.split(r'[。.!?！？]', text) if p]


def fixed_chunks(text, size=34, overlap=8):
    out = []
    i = 0
    while i < len(text):
        out.append(text[i:i+size])
        if i + size >= len(text):
            break
        i += size - overlap
    return out

chunks = []
for doc in documents:
    for mode, parts in [('sentence', sentence_chunks(doc['text'])), ('fixed', fixed_chunks(doc['text']))]:
        for idx, part in enumerate(parts):
            chunks.append({
                'chunk_id': f"{doc['id']}-{mode[0]}{idx}",
                'doc_id': doc['id'],
                'title': doc['title'],
                'mode': mode,
                'text': part,
            })

print('documents:', len(documents))
print('chunks:', len(chunks))
for c in chunks[:4]:
    print(c['chunk_id'], c['mode'], c['text'])

## TF-IDF で検索器を作る

埋め込みモデルを使わなくても、検索器の構造は確認できます。各チャンクを文字 2-gram の TF-IDF ベクトルに変換し、質問ベクトルとの cosine 類似度で並べます。

In [ ]:
def build_index(chunks):
    tokenized = [tokenize(c['title'] + c['text']) for c in chunks]
    df = Counter()
    for toks in tokenized:
        df.update(set(toks))
    n = len(chunks)
    idf = {tok: math.log((1 + n) / (1 + freq)) + 1.0 for tok, freq in df.items()}
    vectors = []
    for toks in tokenized:
        tf = Counter(toks)
        vec = {tok: count * idf[tok] for tok, count in tf.items()}
        vectors.append(vec)
    return {'chunks': chunks, 'idf': idf, 'vectors': vectors}


def vectorize_query(query, index):
    tf = Counter(tokenize(query))
    return {tok: count * index['idf'][tok] for tok, count in tf.items() if tok in index['idf']}


def retrieve(query, index, top_k=5):
    qv = vectorize_query(query, index)
    scored = []
    for chunk, vec in zip(index['chunks'], index['vectors']):
        scored.append({**chunk, 'score': cosine(qv, vec)})
    scored.sort(key=lambda x: x['score'], reverse=True)
    return scored[:top_k]

index = build_index(chunks)
for hit in retrieve('LoRAの利点を説明して', index, top_k=4):
    print(hit['chunk_id'], round(hit['score'], 4), hit['text'])

## 再ランキングと重複除去を挟む

初回検索の上位には、同じ文書から切り方違いのチャンクが複数入ることがあります。タイトル一致、本文一致、検索スコアを合わせて並べ直し、同じ文を何度も文脈に入れないようにします。

再ランキングは、検索器そのものを置き換える処理ではありません。粗く拾った候補を、質問語、タイトル、本文の対応で並べ直す後段処理です。運用では学習済み reranker やクリックログを使うこともあります。

In [ ]:
def rerank(query, hits):
    q_terms = set(tokenize(query))
    out = []
    for h in hits:
        text_terms = set(tokenize(h['text']))
        title_terms = set(tokenize(h['title']))
        text_overlap = len(q_terms & text_terms) / max(len(q_terms), 1)
        title_overlap = len(q_terms & title_terms) / max(len(q_terms), 1)
        rerank_score = 0.70 * h['score'] + 0.20 * text_overlap + 0.10 * title_overlap
        out.append({**h, 'rerank_score': rerank_score})
    out.sort(key=lambda x: x['rerank_score'], reverse=True)
    return out


def retrieve_context(query, top_k=3):
    raw = retrieve(query, index, top_k=top_k * 4)
    ranked = rerank(query, raw)
    selected = []
    seen = set()
    for item in ranked:
        key = (item['doc_id'], item['text'])
        if key in seen:
            continue
        seen.add(key)
        selected.append(item)
        if len(selected) == top_k:
            break
    return selected

ctx = retrieve_context('RAGの評価で分けるべき段階は?', top_k=3)
for item in ctx:
    print(item['chunk_id'], round(item['rerank_score'], 4), item['text'])

## 引用付き回答を作る

回答は取得文脈に支えられている必要があります。自由生成の代わりに、質問と重なりが最大の文を抽出し、参照 ID を付けます。実システムでは生成モデルを使っても、参照 ID と回答文の対応を検査します。

In [ ]:
def answer_with_citations(query, context):
    q_terms = set(tokenize(query))
    best = None
    for item in context:
        overlap = len(q_terms & set(tokenize(item['text'])))
        score = overlap + 0.25 * item.get('rerank_score', 0.0)
        if best is None or score > best['score']:
            best = {'score': score, 'item': item}
    if best is None or best['score'] <= 0:
        return {'answer': '該当する根拠が見つかりません。', 'refs': [], 'context': context}
    item = best['item']
    return {
        'answer': item['text'],
        'refs': [f"[{item['doc_id']}:{item['chunk_id']}]"] ,
        'context': context,
    }

rag_query = 'ベルマン最適方程式とは何か'
rag_result = answer_with_citations(rag_query, retrieve_context(rag_query, top_k=3))
print(rag_result['answer'])
print('refs:', rag_result['refs'])

## 検索品質と回答品質を分けて測る

検索が外れると、どれほど流暢に答えても根拠は崩れます。回答が外れる場合でも、検索は合っていたのか、引用の対応が悪かったのかで改善策が変わります。

hit@k は正しい文書が候補に入ったかを見ます。MRR は正しい文書が何位に来たかを見ます。引用対応は、回答文が参照チャンクから支えられているかを見ます。

In [ ]:
def retrieval_metrics(tests, k=3):
    hit1 = 0
    hitk = 0
    mrr = 0.0
    for query, expected_doc in tests:
        hits = retrieve_context(query, top_k=k)
        docs = [h['doc_id'] for h in hits]
        hit1 += int(docs and docs[0] == expected_doc)
        hitk += int(expected_doc in docs)
        rank = next((i for i, doc_id in enumerate(docs, start=1) if doc_id == expected_doc), None)
        mrr += 0.0 if rank is None else 1.0 / rank
    n = len(tests)
    return {'hit@1': hit1 / n, f'hit@{k}': hitk / n, 'mrr': mrr / n}


def citation_overlap(answer, refs):
    support_text = []
    lookup = {(c['doc_id'], c['chunk_id']): c['text'] for c in chunks}
    for ref in refs:
        m = re.match(r'^\[(.+?):(.+?)\]$', ref)
        if m:
            support_text.append(lookup.get((m.group(1), m.group(2)), ''))
    a_terms = set(tokenize(answer))
    s_terms = set(tokenize(''.join(support_text)))
    return len(a_terms & s_terms) / max(len(a_terms), 1)

tests = [
    ('ベルマン方程式を説明して', 'rl'),
    ('LoRAは何を更新するか', 'finetune'),
    ('RAGでは何を分けて評価するか', 'rag'),
    ('Input Railsの役割は?', 'guardrails'),
]
print(retrieval_metrics(tests, k=3))
print('citation overlap:', round(citation_overlap(rag_result['answer'], rag_result['refs']), 3))

## Tool Use: 安全な計算ツール

数式計算は検索より計算機へ渡した方が正確です。ツールは入力制限を持つ必要があります。下の計算器は数値、四則演算、べき乗だけを許可し、関数呼び出しや変数参照を拒否します。

In [ ]:
BIN_OPS = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv, ast.Pow: op.pow}
UNARY_OPS = {ast.UAdd: op.pos, ast.USub: op.neg}


def eval_node(node):
    if isinstance(node, ast.Expression):
        return eval_node(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)
    if isinstance(node, ast.UnaryOp) and type(node.op) in UNARY_OPS:
        return UNARY_OPS[type(node.op)](eval_node(node.operand))
    if isinstance(node, ast.BinOp) and type(node.op) in BIN_OPS:
        left = eval_node(node.left)
        right = eval_node(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 8:
            raise ValueError('exponent too large')
        return BIN_OPS[type(node.op)](left, right)
    raise ValueError('unsupported expression')


def calculator(expression):
    try:
        tree = ast.parse(expression, mode='eval')
        return {'ok': True, 'value': eval_node(tree)}
    except Exception as exc:
        return {'ok': False, 'error': str(exc)}

for expr in ['(12 + 8) / 5', '2 ** 6', '__import__("os").system("ls")']:
    print(expr, '->', calculator(expr))

## Tool Use: HTML から操作候補を取る

Web 操作では、ページの DOM からボタンやリンクを読み、指示と合う候補を選びます。実行前には危険操作を止め、候補をユーザー確認へ回します。

In [ ]:
class ActionParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.stack = []
        self.actions = []

    def handle_starttag(self, tag, attrs):
        self.stack.append({'tag': tag, 'attrs': dict(attrs), 'text': []})

    def handle_data(self, data):
        text = data.strip()
        if text:
            for node in self.stack:
                node['text'].append(text)

    def handle_endtag(self, tag):
        if not self.stack:
            return
        node = self.stack.pop()
        if node['tag'] == tag and tag in {'button', 'a'}:
            attrs = node['attrs']
            self.actions.append({
                'tag': tag,
                'id': attrs.get('id', ''),
                'href': attrs.get('href', ''),
                'label': ' '.join(node['text']) or attrs.get('aria-label', '') or attrs.get('title', ''),
            })


def extract_actions(html):
    parser = ActionParser()
    parser.feed(html)
    return parser.actions


def choose_action(html, instruction):
    risky = {'削除', 'delete', '送金', 'purchase', 'buy'}
    if any(word in instruction.lower() for word in risky):
        return {'action': 'blocked', 'reason': 'risky instruction'}
    terms = set(tokenize(instruction))
    candidates = extract_actions(html)
    scored = []
    for action in candidates:
        hay = action['label'] + action['id'] + action['href']
        score = len(terms & set(tokenize(hay)))
        scored.append((score, action))
    scored.sort(key=lambda x: x[0], reverse=True)
    if not scored or scored[0][0] == 0:
        return {'action': 'none', 'reason': 'no matching element'}
    return {'action': 'click_candidate', 'target': scored[0][1], 'score': scored[0][0]}

html = '<main><button id="save">保存</button><button id="delete">削除</button><a href="/docs">資料を見る</a></main>'
print(extract_actions(html))
print(choose_action(html, '資料を開いて'))
print(choose_action(html, '削除して'))

## ルーティングで道具を選ぶ

入力に応じて、検索、計算、Web 操作を切り替えます。ルールベースでも、ログに plan と tool_output を残すと、どこで失敗したかを追跡しやすくなります。

ルーティングは精度だけでなく安全性にも関わります。計算式を検索へ送ると不安定になり、危険な操作を Web ツールへ直接送ると事故につながります。plan を残す理由は、後から判断根拠を追うためです。

In [ ]:
def route(query):
    has_date = re.search(r'\b\d{4}-\d{1,2}-\d{1,2}\b', query) is not None
    has_digit = re.search(r'\d', query) is not None
    has_operator = any(op_char in query for op_char in ['+', '-', '*', '/', '^'])
    if has_digit and has_operator and not has_date:
        expr = query.replace('^', '**')
        expr = re.sub(r'[^0-9+\-*/(). *]', '', expr)
        return {'tool': 'calculator', 'args': {'expression': expr}}
    if any(word in query for word in ['クリック', '開いて', '保存', '削除']):
        return {'tool': 'web', 'args': {'instruction': query}}
    return {'tool': 'retrieval', 'args': {'query': query}}

def orchestrate(query, html_context=''):
    plan = route(query)
    if plan['tool'] == 'calculator':
        output = calculator(plan['args']['expression'])
        final = f"計算結果: {output.get('value', output.get('error'))}"
    elif plan['tool'] == 'web':
        output = choose_action(html_context, plan['args']['instruction'])
        if output['action'] == 'click_candidate':
            t = output['target']
            final = f"操作候補: click id={t['id']} label={t['label']}"
        else:
            final = output.get('reason', 'no action')
    else:
        context = retrieve_context(plan['args']['query'], top_k=3)
        output = answer_with_citations(query, context)
        final = output['answer'] + ' ' + ' '.join(output['refs'])
    return {'plan': plan, 'tool_output': output, 'final': final}

for query in ['LoRAは何を更新する?', '12 * (3 + 4) は?', '資料を開いて']:
    result = orchestrate(query, html_context=html)
    print(query)
    print(' plan:', result['plan'])
    print(' final:', result['final'])

## 評価は段階ごとに分ける

RAG は hit@k と MRR、引用対応、回答品質を分けます。Tool Use は routing accuracy、tool success、end-to-end success を分けます。すべてを最終回答の良し悪しだけで見ると、検索を直すべきか、生成を直すべきか、ツールの失敗処理を直すべきかが分かりません。

In [ ]:
routing_tests = [
    ('3 + 5 * 2', 'calculator'),
    ('ベルマン方程式とは?', 'retrieval'),
    ('資料を開いて', 'web'),
    ('Input Railsの役割は?', 'retrieval'),
]
correct = 0
for query, expected in routing_tests:
    actual = route(query)['tool']
    correct += int(actual == expected)
    print(query, 'expected=', expected, 'actual=', actual)
print('routing accuracy:', round(correct / len(routing_tests), 3))

cost = {
    'requests_per_day': 900,
    'input_tokens': 1200,
    'output_tokens': 180,
    'input_usd_per_million': 0.20,
    'output_usd_per_million': 0.80,
}
per_request = cost['input_tokens'] / 1_000_000 * cost['input_usd_per_million'] + cost['output_tokens'] / 1_000_000 * cost['output_usd_per_million']
print('estimated daily cost USD:', round(per_request * cost['requests_per_day'], 4))

RAG と Tool Use は、外部の情報や能力をモデルへ接続するための設計です。RAG は根拠を取りに行き、Tool Use は操作や計算を外へ委譲します。実装では、チャンク化、検索、再ランキング、引用、ルーティング、ツール実行、失敗時の停止を分け、各段階のログと指標を残します。